# CICIoT2023 DDoS 기기별 EDA — 간소화 버전

이미 생성한 `analysis_results/all_devices_summary.csv`만 사용합니다.

**이 노트북에서는 PCAP을 다시 읽지 않습니다.**

실행 순서:
1. CSV 불러오기
2. MAC별 기기/역할 정보 붙이기
3. 12개 공격별 송신·수신·전체 TOP 5 및 주요 피해 후보 확인
4. Camera 분석
5. 다음 실험용 CSV 저장


In [1]:
# 1. Google Drive 연결 + 저장된 EDA 결과 불러오기
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/CICIoT2023')
RESULT_DIR = BASE / 'analysis_results'
CSV_PATH = RESULT_DIR / 'all_devices_summary.csv'

if not CSV_PATH.exists():
    raise FileNotFoundError(f'파일이 없습니다: {CSV_PATH}')

df = pd.read_csv(CSV_PATH)

# MAC 표기 통일
for col in ['mac', 'top_peer']:
    if col in df.columns:
        df[col] = df[col].astype('string').str.strip().str.upper()

df['attack'] = df['file'].str.replace('.pcap', '', regex=False)

print('행 수:', len(df))
print('공격 파일 수:', df['file'].nunique())
display(df.head())


Mounted at /content/drive
행 수: 1233
공격 파일 수: 12


,mac,ips,sent_packets,received_packets,total_packets,sent_bytes,received_bytes,total_bytes,unique_peers,top_peer,top_peer_packets,file,attack
0,B0:C5:54:59:2E:99,192.168.137.132,8278,878305,886583,516304,826686739,827203043,9,DC:A6:32:C9:E4:AB,221817,DDoS-ACK_Fragmentation.pcap,DDoS-ACK_Fragmentation
1,8C:85:80:6C:B6:47,"192.168.137.123, fe80::8e85:80ff:fe6c:b647",268504,544412,812916,16137169,512315019,528452188,11,DC:A6:32:DC:27:D5,339968,DDoS-ACK_Fragmentation.pcap,DDoS-ACK_Fragmentation
2,DC:A6:32:C9:E4:AB,"0.0.0.0, 192.168.137.196, fd5a:7bd:2aea:e111:d...",555586,56391,611977,522916337,3384066,526300403,9,B0:C5:54:59:2E:99,221817,DDoS-ACK_Fragmentation.pcap,DDoS-ACK_Fragmentation
3,DC:A6:32:DC:27:D5,"0.0.0.0, 192.168.137.65, fd5a:7bd:2aea:e111:de...",486229,114608,600837,457526433,6877616,464404049,9,8C:85:80:6C:B6:47,339968,DDoS-ACK_Fragmentation.pcap,DDoS-ACK_Fragmentation
4,9C:8E:CD:1D:AB:9F,192.168.137.120,12678,564673,577351,1181940,530046438,531228378,9,DC:A6:32:C9:E4:AB,206525,DDoS-ACK_Fragmentation.pcap,DDoS-ACK_Fragmentation


In [2]:
# 2. CICIoT2023에서 현재 분석에 필요한 기기 정보만 등록

DDOS_ATTACKERS = {
    'E4:5F:01:55:90:C4': 'Raspberry Pi 4 4GB',
    'DC:A6:32:C9:E4:D5': 'Raspberry Pi 4 2GB',
    'DC:A6:32:DC:27:D5': 'Raspberry Pi 4 8GB',
    'DC:A6:32:C9:E5:EF': 'Raspberry Pi 4 2GB',
    'DC:A6:32:C9:E4:AB': 'Raspberry Pi 4 2GB',
    'DC:A6:32:C9:E4:90': 'Raspberry Pi 4 2GB',
    'DC:A6:32:C9:E5:A4': 'Raspberry Pi 4 2GB',
}

CAMERAS = {
    '9C:8E:CD:1D:AB:9F': 'AMCREST WiFi Camera',
    '3C:37:86:6F:B9:51': 'Arlo Base Station',
    '40:5D:82:35:14:C8': 'Arlo Q Indoor Camera',
    'C0:E7:BF:0A:79:D1': 'Borun/Sichuan-AI Camera',
    'B0:C5:54:59:2E:99': 'D-Link Mini Camera',
    '44:01:BB:EC:10:4A': 'HeimVision Smart WiFi Camera',
    '34:75:63:73:F3:36': 'Home Eye Camera',
    '7C:A7:B0:CD:18:32': 'Luohe Cam Dog Camera',
    '44:BB:3B:00:39:07': 'Nest Indoor Camera',
    '70:EE:50:68:0E:32': 'Netatmo Camera',
    '10:5A:17:97:A5:C6': 'Rbcior Camera',
    '10:2C:6B:1B:43:BE': 'SIMCAM 1S',
    '6C:5A:B0:44:1D:90': 'TP-Link Tapo Camera',
    '7C:78:B2:86:0D:81': 'Wyze Camera',
    '84:7A:B6:64:62:58': 'Yi Indoor Camera',
    '84:7A:B6:62:3A:6C': 'Yi Indoor 2 Camera',
    '2C:D2:6B:66:D2:87': 'Yi Outdoor Camera',
}

KNOWN_IOT = {
    '1C:12:B0:9B:0C:EC': 'Amazon Alexa Echo Spot',
    '1C:FE:2B:98:16:DD': 'Amazon Alexa Echo Dot 1',
    'CC:F4:11:9C:D0:00': 'Google Nest Mini Speaker',
    **CAMERAS,
}

GATEWAY_CANDIDATES = {'3C:18:A0:41:C3:A0'}

DEVICE_NAMES = {**KNOWN_IOT, **DDOS_ATTACKERS}

def is_group_mac(mac):
    if pd.isna(mac):
        return False
    try:
        return bool(int(str(mac).split(':')[0], 16) & 1)
    except:
        return False

def get_role(mac):
    if pd.isna(mac):
        return 'unknown'
    if is_group_mac(mac):
        return 'broadcast/multicast'
    if mac in DDOS_ATTACKERS:
        return 'DDoS attacker'
    if mac in CAMERAS:
        return 'Camera'
    if mac in KNOWN_IOT:
        return 'IoT device'
    if mac in GATEWAY_CANDIDATES:
        return 'network/gateway candidate'
    return 'unknown'

df['device_name'] = df['mac'].map(DEVICE_NAMES).fillna(df['mac'])
df['role'] = df['mac'].apply(get_role)
df['top_peer_name'] = df['top_peer'].map(DEVICE_NAMES).fillna(df['top_peer'])
df['top_peer_role'] = df['top_peer'].apply(get_role)

# 방향성 확인용 지표
df['received_to_sent_ratio'] = (
    (df['received_packets'] + 1) / (df['sent_packets'] + 1)
)

# 실제 기기가 아닌 broadcast/multicast는 순위에서 제외
eda = df[df['role'] != 'broadcast/multicast'].copy()

display(
    eda[['attack', 'device_name', 'mac', 'role',
         'sent_packets', 'received_packets', 'total_packets',
         'top_peer_name', 'top_peer_role']]
    .sort_values('received_packets', ascending=False)
    .head(20)
)


,attack,device_name,mac,role,sent_packets,received_packets,total_packets,top_peer_name,top_peer_role
227,DDoS-ICMP_Flood,Amazon Alexa Echo Dot 1,1C:FE:2B:98:16:DD,IoT device,2280,15787965,15790245,Raspberry Pi 4 4GB,DDoS attacker
919,DDoS-TCP_Flood,Amazon Alexa Echo Dot 1,1C:FE:2B:98:16:DD,IoT device,3833,14702613,14706446,Raspberry Pi 4 8GB,DDoS attacker
1024,DDoS-UDP_Flood,Amazon Alexa Echo Dot 1,1C:FE:2B:98:16:DD,IoT device,7075,13541168,13548243,Raspberry Pi 4 4GB,DDoS attacker
429,DDoS-PSHACK_Flood,Amazon Alexa Echo Dot 1,1C:FE:2B:98:16:DD,IoT device,187054,12994828,13181882,Raspberry Pi 4 2GB,DDoS attacker
831,DDoS-SynonymousIP_Flood,2C:71:FF:05:F1:15,2C:71:FF:05:F1:15,unknown,2546,12909807,12912353,Raspberry Pi 4 2GB,DDoS attacker
530,DDoS-RSTFINFlood,Amazon Alexa Echo Dot 1,1C:FE:2B:98:16:DD,IoT device,2363,12575492,12577855,Raspberry Pi 4 8GB,DDoS attacker
832,DDoS-SynonymousIP_Flood,AMCREST WiFi Camera,9C:8E:CD:1D:AB:9F,Camera,1054,12032654,12033708,Raspberry Pi 4 2GB,DDoS attacker
531,DDoS-RSTFINFlood,28:6D:97:9E:F4:D5,28:6D:97:9E:F4:D5,unknown,1056,11989659,11990715,Raspberry Pi 4 8GB,DDoS attacker
430,DDoS-PSHACK_Flood,Amazon Alexa Echo Spot,1C:12:B0:9B:0C:EC,IoT device,164314,11382204,11546518,Raspberry Pi 4 2GB,DDoS attacker
228,DDoS-ICMP_Flood,Amazon Alexa Echo Spot,1C:12:B0:9B:0C:EC,IoT device,9004,10893507,10902511,Raspberry Pi 4 8GB,DDoS attacker


In [3]:
# 3. 12개 공격별 핵심 요약
#    - 송신 TOP 5
#    - 수신 TOP 5
#    - 전체 통신량 TOP 5
#    - 공격자/gateway를 제외한 주요 수신 기기
#    - 가장 많이 수신한 Camera

def top5_text(g, column):
    rows = g.nlargest(5, column)
    return ' | '.join(
        f'{name} ({value:,})'
        for name, value in zip(rows['device_name'], rows[column].astype(int))
    )

summary_rows = []

for attack, g in eda.groupby('attack', sort=True):
    victim_pool = g[
        ~g['role'].isin(['DDoS attacker', 'network/gateway candidate'])
    ]
    main_victim = (
        victim_pool.nlargest(1, 'received_packets').iloc[0]
        if len(victim_pool) else None
    )

    cams = g[g['role'] == 'Camera']
    main_cam = (
        cams.nlargest(1, 'received_packets').iloc[0]
        if len(cams) else None
    )

    summary_rows.append({
        'attack': attack,
        'sent_TOP5': top5_text(g, 'sent_packets'),
        'received_TOP5': top5_text(g, 'received_packets'),
        'total_TOP5': top5_text(g, 'total_packets'),
        'major_received_device': None if main_victim is None else main_victim['device_name'],
        'major_received_packets': None if main_victim is None else int(main_victim['received_packets']),
        'top_camera': None if main_cam is None else main_cam['device_name'],
        'camera_received': None if main_cam is None else int(main_cam['received_packets']),
        'camera_sent': None if main_cam is None else int(main_cam['sent_packets']),
        'camera_top_peer': None if main_cam is None else main_cam['top_peer_name'],
    })

attack_summary = pd.DataFrame(summary_rows)

display(attack_summary)


,attack,sent_TOP5,received_TOP5,total_TOP5,major_received_device,major_received_packets,top_camera,camera_received,camera_sent,camera_top_peer
0,DDoS-ACK_Fragmentation,"Raspberry Pi 4 2GB (555,586) | Raspberry Pi 4 ...","D-Link Mini Camera (878,305) | AMCREST WiFi Ca...","D-Link Mini Camera (886,583) | 8C:85:80:6C:B6:...",D-Link Mini Camera,878305,D-Link Mini Camera,878305,8278,Raspberry Pi 4 2GB
1,DDoS-HTTP_Flood-,"B0:F1:EC:D3:E7:98 (341,724) | DC:A6:32:C9:E5:0...","AMCREST WiFi Camera (475,270) | Raspberry Pi 4...","AMCREST WiFi Camera (687,398) | Raspberry Pi 4...",AMCREST WiFi Camera,475270,AMCREST WiFi Camera,475270,212128,Raspberry Pi 4 8GB
2,DDoS-ICMP_Flood,"Raspberry Pi 4 8GB (4,753,338) | Raspberry Pi ...","Amazon Alexa Echo Dot 1 (15,787,965) | Amazon ...","Amazon Alexa Echo Dot 1 (15,790,245) | Amazon ...",Amazon Alexa Echo Dot 1,15787965,Nest Indoor Camera,1870,2254,3C:18:A0:41:C3:A0
3,DDoS-ICMP_Fragmentation,"Raspberry Pi 4 2GB (394,462) | Raspberry Pi 4 ...","Amazon Alexa Echo Dot 1 (790,754) | Google Nes...","Amazon Alexa Echo Dot 1 (794,383) | Google Nes...",Amazon Alexa Echo Dot 1,790754,AMCREST WiFi Camera,2234,3181,3C:18:A0:41:C3:A0
4,DDoS-PSHACK_Flood,"Raspberry Pi 4 2GB (5,235,887) | Raspberry Pi ...","Amazon Alexa Echo Dot 1 (12,994,828) | Amazon ...","Amazon Alexa Echo Dot 1 (13,181,882) | Amazon ...",Amazon Alexa Echo Dot 1,12994828,AMCREST WiFi Camera,1960013,5119,Raspberry Pi 4 8GB
5,DDoS-RSTFINFlood,"Raspberry Pi 4 8GB (4,745,664) | Raspberry Pi ...","Amazon Alexa Echo Dot 1 (12,575,492) | 28:6D:9...","Amazon Alexa Echo Dot 1 (12,577,855) | 28:6D:9...",Amazon Alexa Echo Dot 1,12575492,Arlo Q Indoor Camera,4620,4126,3C:18:A0:41:C3:A0
6,DDoS-SYN_Flood,"Raspberry Pi 4 2GB (7,079,462) | Raspberry Pi ...","Amazon Alexa Echo Spot (8,786,693) | Amazon Al...","Amazon Alexa Echo Spot (8,957,322) | Amazon Al...",Amazon Alexa Echo Spot,8786693,AMCREST WiFi Camera,2870769,14686,Raspberry Pi 4 2GB
7,DDoS-SlowLoris,"3C:18:A0:41:C3:A0 (376,053) | Arlo Q Indoor Ca...","3C:18:A0:41:C3:A0 (411,558) | Arlo Q Indoor Ca...","3C:18:A0:41:C3:A0 (787,611) | Arlo Q Indoor Ca...",Arlo Q Indoor Camera,162541,Arlo Q Indoor Camera,162541,183985,3C:18:A0:41:C3:A0
8,DDoS-SynonymousIP_Flood,"Raspberry Pi 4 2GB (7,855,183) | Raspberry Pi ...","2C:71:FF:05:F1:15 (12,909,807) | AMCREST WiFi ...","2C:71:FF:05:F1:15 (12,912,353) | AMCREST WiFi ...",2C:71:FF:05:F1:15,12909807,AMCREST WiFi Camera,12032654,1054,Raspberry Pi 4 2GB
9,DDoS-TCP_Flood,"Raspberry Pi 4 8GB (6,170,232) | Raspberry Pi ...","Amazon Alexa Echo Dot 1 (14,702,613) | Amazon ...","Amazon Alexa Echo Dot 1 (14,706,446) | Amazon ...",Amazon Alexa Echo Dot 1,14702613,Wyze Camera,6126,6889,3C:18:A0:41:C3:A0


In [4]:
# 4. Camera만 따로 분석

camera_df = (
    eda[eda['role'] == 'Camera']
    [[
        'attack', 'device_name', 'mac', 'ips',
        'sent_packets', 'received_packets', 'total_packets',
        'received_to_sent_ratio',
        'top_peer_name', 'top_peer_role', 'top_peer_packets'
    ]]
    .sort_values(['attack', 'received_packets'], ascending=[True, False])
    .copy()
)

# 각 Camera가 12개 공격 전체에서 받은 트래픽 합계
camera_total = (
    camera_df
    .groupby(['device_name', 'mac'], as_index=False)
    .agg(
        sent_packets=('sent_packets', 'sum'),
        received_packets=('received_packets', 'sum'),
        total_packets=('total_packets', 'sum')
    )
)

camera_total['received_to_sent_ratio'] = (
    (camera_total['received_packets'] + 1) /
    (camera_total['sent_packets'] + 1)
)

# 각 Camera가 가장 많은 패킷을 받은 공격
if len(camera_df):
    peak_idx = camera_df.groupby('mac')['received_packets'].idxmax()
    peak = (
        camera_df.loc[peak_idx, ['mac', 'attack', 'received_packets']]
        .rename(columns={
            'attack': 'peak_attack',
            'received_packets': 'peak_attack_received'
        })
    )
    camera_total = camera_total.merge(peak, on='mac', how='left')

camera_total = camera_total.sort_values('received_packets', ascending=False)

print('공격별 Camera 결과')
display(camera_df)

print('\n12개 공격 전체 Camera 누적 결과')
display(camera_total)


공격별 Camera 결과


,attack,device_name,mac,ips,sent_packets,received_packets,total_packets,received_to_sent_ratio,top_peer_name,top_peer_role,top_peer_packets
0,DDoS-ACK_Fragmentation,D-Link Mini Camera,B0:C5:54:59:2E:99,192.168.137.132,8278,878305,886583,106.088416,Raspberry Pi 4 2GB,DDoS attacker,221817
4,DDoS-ACK_Fragmentation,AMCREST WiFi Camera,9C:8E:CD:1D:AB:9F,192.168.137.120,12678,564673,577351,44.536162,Raspberry Pi 4 2GB,DDoS attacker,206525
16,DDoS-ACK_Fragmentation,Arlo Q Indoor Camera,40:5D:82:35:14:C8,192.168.137.51,7020,6290,13310,0.896026,3C:18:A0:41:C3:A0,network/gateway candidate,13169
17,DDoS-ACK_Fragmentation,Nest Indoor Camera,44:BB:3B:00:39:07,"0.0.0.0, 192.168.137.149",3177,2834,6011,0.892070,3C:18:A0:41:C3:A0,network/gateway candidate,5951
21,DDoS-ACK_Fragmentation,Netatmo Camera,70:EE:50:68:0E:32,"0.0.0.0, 192.168.137.48, ::, fe80::72ee:50ff:f...",2880,2163,5043,0.751128,3C:18:A0:41:C3:A0,network/gateway candidate,4628
...,...,...,...,...,...,...,...,...,...,...,...
1172,DDoS-UDP_Fragmentation,Yi Indoor Camera,84:7A:B6:64:62:58,"0.0.0.0, 192.168.137.95",552,511,1063,0.925859,3C:18:A0:41:C3:A0,network/gateway candidate,998
1174,DDoS-UDP_Fragmentation,Yi Indoor 2 Camera,84:7A:B6:62:3A:6C,"0.0.0.0, 192.168.137.53",485,459,944,0.946502,3C:18:A0:41:C3:A0,network/gateway candidate,890
1179,DDoS-UDP_Fragmentation,D-Link Mini Camera,B0:C5:54:59:2E:99,"0.0.0.0, 192.168.137.132",378,388,766,1.026385,3C:18:A0:41:C3:A0,network/gateway candidate,648
1178,DDoS-UDP_Fragmentation,Arlo Base Station,3C:37:86:6F:B9:51,"192.168.137.63, fe80::3e37:86ff:fe6f:b951",448,350,798,0.781737,3C:18:A0:41:C3:A0,network/gateway candidate,756



12개 공격 전체 Camera 누적 결과


,device_name,mac,sent_packets,received_packets,total_packets,received_to_sent_ratio,peak_attack,peak_attack_received
0,AMCREST WiFi Camera,9C:8E:CD:1D:AB:9F,298156,18135242,18433398,60.824475,DDoS-SynonymousIP_Flood,12032654
4,D-Link Mini Camera,B0:C5:54:59:2E:99,333908,1256519,1590427,3.763061,DDoS-ACK_Fragmentation,878305
2,Arlo Q Indoor Camera,40:5D:82:35:14:C8,226216,195213,421429,0.862950,DDoS-SlowLoris,162541
12,TP-Link Tapo Camera,6C:5A:B0:44:1D:90,154972,153983,308955,0.993618,DDoS-SlowLoris,141488
5,HeimVision Smart WiFi Camera,44:01:BB:EC:10:4A,98933,95905,194838,0.969394,DDoS-SlowLoris,90285
8,Nest Indoor Camera,44:BB:3B:00:39:07,93274,85412,178686,0.915712,DDoS-SlowLoris,66702
9,Netatmo Camera,70:EE:50:68:0E:32,203939,49768,253707,0.244037,DDoS-SlowLoris,40167
13,Wyze Camera,7C:78:B2:86:0D:81,35793,28691,64484,0.801587,DDoS-SlowLoris,9038
6,Home Eye Camera,34:75:63:73:F3:36,17040,12574,29614,0.737926,DDoS-SlowLoris,4944
11,SIMCAM 1S,10:2C:6B:1B:43:BE,14324,10710,25034,0.747714,DDoS-SYN_Flood,2926


In [5]:
# 5. 다음 분석에서 바로 재사용하도록 결과 저장

attack_summary.to_csv(
    RESULT_DIR / 'attack_device_summary.csv',
    index=False
)

camera_df.to_csv(
    RESULT_DIR / 'camera_by_attack.csv',
    index=False
)

camera_total.to_csv(
    RESULT_DIR / 'camera_total_summary.csv',
    index=False
)

print('저장 완료:')
print(RESULT_DIR / 'attack_device_summary.csv')
print(RESULT_DIR / 'camera_by_attack.csv')
print(RESULT_DIR / 'camera_total_summary.csv')


저장 완료:
/content/drive/MyDrive/CICIoT2023/analysis_results/attack_device_summary.csv
/content/drive/MyDrive/CICIoT2023/analysis_results/camera_by_attack.csv
/content/drive/MyDrive/CICIoT2023/analysis_results/camera_total_summary.csv


In [8]:
device_total = (
    eda
    .groupby(['device_name', 'mac', 'role'], as_index=False)
    .agg(
        sent_packets=('sent_packets', 'sum'),
        received_packets=('received_packets', 'sum'),
        total_packets=('total_packets', 'sum')
    )
    .sort_values('total_packets', ascending=False)
)

display(device_total.head(20))

,device_name,mac,role,sent_packets,received_packets,total_packets
35,Amazon Alexa Echo Dot 1,1C:FE:2B:98:16:DD,IoT device,455769,78796257,79252026
36,Amazon Alexa Echo Spot,1C:12:B0:9B:0C:EC,IoT device,395313,48173092,48568405
75,Raspberry Pi 4 2GB,DC:A6:32:C9:E4:AB,DDoS attacker,34556665,552436,35109101
80,Raspberry Pi 4 8GB,DC:A6:32:DC:27:D5,DDoS attacker,29834591,742899,30577490
74,Raspberry Pi 4 2GB,DC:A6:32:C9:E4:90,DDoS attacker,26893311,525248,27418559
76,Raspberry Pi 4 2GB,DC:A6:32:C9:E4:D5,DDoS attacker,26468797,434400,26903197
79,Raspberry Pi 4 4GB,E4:5F:01:55:90:C4,DDoS attacker,25870427,480346,26350773
78,Raspberry Pi 4 2GB,DC:A6:32:C9:E5:EF,DDoS attacker,25314686,447246,25761932
77,Raspberry Pi 4 2GB,DC:A6:32:C9:E5:A4,DDoS attacker,23978107,487029,24465136
34,AMCREST WiFi Camera,9C:8E:CD:1D:AB:9F,Camera,298156,18135242,18433398
